In [1]:
!pip install fastparquet pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.3 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

# Then move kaggle.json into the folder where the API expects to find it.
! mkdir -p ~/.kaggle/ && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json
User uploaded file "kaggle.json" with length 69 bytes


In [3]:
# Get Titanic Dataset
!kaggle datasets download -d wordsforthewise/lending-club

Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
License(s): CC0-1.0
100% 1.26G/1.26G [00:11<00:00, 80.6MB/s]
100% 1.26G/1.26G [00:11<00:00, 123MB/s] 


In [4]:
!ls -lah

total 1.3G
drwxr-xr-x 1 root root 4.0K Jun  7 04:44 .
drwxr-xr-x 1 root root 4.0K Jun  7 04:41 ..
drwxr-xr-x 4 root root 4.0K Jun  5 13:38 .config
-rw-r--r-- 1 root root 1.3G Dec 17  2019 lending-club.zip
drwxr-xr-x 1 root root 4.0K Jun  5 13:38 sample_data


In [5]:
# GZIP 압축파일 375MB: accepted_2007_to_2018Q4.csv.gz
# GZIP 압축파일 244MB: rejected_2007_to_2018Q4.csv.gz
!unzip lending-club.zip
!ls -lah

Archive:  lending-club.zip
  inflating: accepted_2007_to_2018Q4.csv.gz  
  inflating: accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv  
  inflating: rejected_2007_to_2018Q4.csv.gz  
  inflating: rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv  
total 1.9G
drwxr-xr-x 1 root root 4.0K Jun  7 04:45 .
drwxr-xr-x 1 root root 4.0K Jun  7 04:41 ..
drwxr-xr-x 2 root root 4.0K Jun  7 04:45 accepted_2007_to_2018q4.csv
-rw-r--r-- 1 root root 375M Dec 17  2019 accepted_2007_to_2018Q4.csv.gz
drwxr-xr-x 4 root root 4.0K Jun  5 13:38 .config
-rw-r--r-- 1 root root 1.3G Dec 17  2019 lending-club.zip
drwxr-xr-x 2 root root 4.0K Jun  7 04:45 rejected_2007_to_2018q4.csv
-rw-r--r-- 1 root root 244M Dec 17  2019 rejected_2007_to_2018Q4.csv.gz
drwxr-xr-x 1 root root 4.0K Jun  5 13:38 sample_data


In [ ]:
# GZIP 압축 해제 후 CSV파일 1.6GB: accepted_2007_to_2018Q4.csv
# GZIP 압축 해제 후 CSV파일 1.7GB: rejected_2007_to_2018Q4.csv
!gzip -d accepted_2007_to_2018Q4.csv.gz
!gzip -d rejected_2007_to_2018Q4.csv.gz
!ls -lah

total 4.5G
drwxr-xr-x 1 root root 4.0K Oct  7 03:03 .
drwxr-xr-x 1 root root 4.0K Oct  7 02:57 ..
drwxr-xr-x 2 root root 4.0K Oct  7 03:02 accepted_2007_to_2018q4.csv
-rw-r--r-- 1 root root 1.6G Dec 17  2019 accepted_2007_to_2018Q4.csv
drwxr-xr-x 4 root root 4.0K Oct  5 19:32 .config
-rw-r--r-- 1 root root 1.3G Dec 17  2019 lending-club.zip
drwxr-xr-x 2 root root 4.0K Oct  7 03:02 rejected_2007_to_2018q4.csv
-rw-r--r-- 1 root root 1.7G Dec 17  2019 rejected_2007_to_2018Q4.csv
drwxr-xr-x 1 root root 4.0K Oct  5 19:33 sample_data


In [ ]:
# accept data, reject_data의 압축 효율 각각 4.3691배, 7.1344배 감축
print("accept_data: {:.4f}, reject_data: {:.4f}".format( 1.6*1024/375, 1.7*1024/244 ))

accept_data: 4.3691, reject_data: 7.1344


In [ ]:
# accept_data 건수:  2,260,702
# reject_data 건수: 27,648,742

# CSV Row 개수 확인하는 명령어
!cat accepted_2007_to_2018Q4.csv | wc -l
!cat rejected_2007_to_2018Q4.csv | wc -l

2260702
27648742


In [ ]:
import pandas as pd
import numpy as np
import gc

In [ ]:
# 압축 시 375MB인 CSV.GZ 파일을 pandas.read_csv로 읽으면 터짐.. => 예상 메모리 약 37GB
# [Error Message]
# <timed exec>:1: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types.
# Specify dtype option on import or set low_memory=False.

%%time
df_accepted = pd.read_csv("/content/accepted_2007_to_2018Q4.csv.gz", compression='gzip')
df_accepted = pd.read_csv("/content/accepted_2007_to_2018Q4.csv")

In [ ]:
# 똑같이 터지겠죠?
%%time
df_rejected = pd.read_csv("/content/rejected_2007_to_2018Q4.csv.gz", compression='gzip')
df_rejected = pd.read_csv("/content/rejected_2007_to_2018Q4.csv")

In [ ]:
# csv.gz file로 accept data 약 226만 건 중 100,000건만 가져와보자.
# 소요시간은? 2.98초
# 메모리는? 0.267 GB

%%time
df_accepted_100000 = pd.read_csv("/content/accepted_2007_to_2018Q4.csv", nrows=100000)

# 메모리 사용량 계산
memory_usage = df_accepted_100000.memory_usage(deep=True).sum() / (1024 ** 3)  # 바이트를 GB로 변환
print(f"DataFrame의 메모리 사용량: {memory_usage:.3f} GB")

del df_accepted_100000
gc.collect()

<timed exec>:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.


DataFrame의 메모리 사용량: 0.267 GB
CPU times: user 2.81 s, sys: 383 ms, total: 3.19 s
Wall time: 3.21 s


0

In [ ]:
# 그렇다면 226만건 데이터는?
# 소요시간은? 67 초
# 메모리는?   6 GB

print("소요시간: ", "%.3f" % (2.98*22.6), "초")
print("메모리:  ", "%.3f" % (0.267*22.6), "GB")

소요시간:  67.348 초
메모리:   6.034 GB


In [ ]:
# csv.gz file로 accept data 약 2764만 건 중 100,000건만 가져와보자.
# 소요시간은? 0.395초
# 메모리는? 0.038 GB

%%time
df_rejected_100000 = pd.read_csv("/content/rejected_2007_to_2018Q4.csv", nrows=100000)

# 메모리 사용량 계산
memory_usage = df_rejected_100000.memory_usage(deep=True).sum() / (1024 ** 3)  # 바이트를 GB로 변환
print(f"DataFrame의 메모리 사용량: {memory_usage:.3f} GB")

del df_rejected_100000
gc.collect()

DataFrame의 메모리 사용량: 0.038 GB
CPU times: user 308 ms, sys: 13.1 ms, total: 321 ms
Wall time: 330 ms


0

In [ ]:
# 그렇다면 2764만건 데이터는?
# 소요시간은?  109 초
# 메모리는?  10.5 GB

print("소요시간: ", "%.3f" % (0.395*276.4), "초")
print("메모리:  ", "%.3f" % (0.038*276.4), "GB")

소요시간:  109.178 초
메모리:   10.503 GB


## 결국 375MB, 244MB로 압축되었던 CSV.GZ 데이터를
## Pandas Dataframe으로 읽으면 메모리만 약 16.5GB를 차지하게 된다??

### 이제부터 CSV를 읽을 때 10만 건씩 Chunk단위로 읽으면서
### Memory Optimize된 DataFrame을 합쳐보도록 하자!

In [ ]:
# 실제 AutoML Solution에 적용되어있는 값비싼 메모리 최적화 소스코드 적용

"""This file contains Data Manager."""

import gc
import numpy
import pandas
import gzip

from pandas.api.types import union_categoricals


class DataTypeChecker:
    """Check the type of data."""

    @staticmethod
    def is_binary(unique_values):
        """Check unique values can be binary."""
        # Remove the NaN and the NaT value from unique_values.
        unique_values_ = {value for value in unique_values if not pandas.isna(value)}
        return len(unique_values_) == 2

    @staticmethod
    def is_integer(unique_values):
        """Check unique values can be integer."""
        # Remove the NaN value from unique values.
        unq_list = [value for value in unique_values if not pandas.isna(value)]
        is_int = (numpy.sum(numpy.round(unq_list) != numpy.array(unq_list)) == 0)
        return is_int

    @staticmethod
    def is_numeric(unique_values):
        """Check unique values can be numeric(float)."""
        # Remove the NaN value from unique values.
        unq_list = [value for value in unique_values if not pandas.isna(value)]
        is_numeric = numpy.array(unq_list).dtype in ['float32', 'float64']
        return is_numeric

    @staticmethod
    def is_multi(unique_values, max_category=200):
        """Check unique values can be multi-category."""
        not_binary = not DataTypeChecker.is_binary(unique_values)
        return 1 < len(unique_values) <= max_category and not_binary

    @staticmethod
    def is_datetime(unique_values):
        """Check unique values can be datetime."""
        if len(unique_values) == 0:
            return False
        if len(unique_values) == 1:
            if pandas.isna(unique_values):
                return False
        for value in unique_values:
            # return false if value is year.
            if pandas.isna(value):
                continue
            if len(str(value)) <= 4:
                return False
        for datetime_case in ['normal', '%Y%m', '%y%m%d', '%m%Y', '%m%d%y']:
            try:
                if datetime_case == 'normal':
                    values = pandas.to_datetime([str(value) for value in unique_values])
                    if not isinstance(values, pandas.DatetimeIndex):
                        return False
                    return True
                else:
                    pandas.to_datetime(
                        [str(value).zfill(6) for value in unique_values],
                        format=datetime_case
                    )
                    return True
            except Exception:
                continue

        return False

    @staticmethod
    def is_nan(unique_values):
        """Check unique values are filled with NaN."""
        if len(unique_values) == 1:
            return pandas.isna(list(unique_values)[0])
        else:
            return False


class DataTypeComparator:
    """Compare the data type."""

    @staticmethod
    def is_datetime(dtype):
        """Compare dtype with numpy.dtype('datetime64[ns]')."""
        return dtype == numpy.dtype('datetime64[ns]')


class PandasOptimizer:
    """Optimize pandas dataframes and series.

    Contains operations on pandas dataframes
    that can be used for optimizations including memory
    allocation optimizations

    TODO: string/object types into category types?
    """

    def __init__(self):
        """Initialize with default variables."""
        self.f32 = numpy.finfo(numpy.float32)
        self.i8 = numpy.iinfo(numpy.int8)
        self.i16 = numpy.iinfo(numpy.int16)
        self.i32 = numpy.iinfo(numpy.int32)
        self.u8 = numpy.iinfo(numpy.uint8)
        self.u16 = numpy.iinfo(numpy.uint16)
        self.u32 = numpy.iinfo(numpy.uint32)

    def _get_max_precision(self, iterable):
        maximum = 0
        for i in iterable:
            try:
                digit_count = len(str(i).split('.')[1])
            except Exception:
                continue
            if maximum < digit_count:
                maximum = digit_count
        return maximum

    def _is_integer(self, iterable):
        for i in iterable:
            if pandas.isnull(i):
                pass
            else:
                if not i.is_integer():
                    return False
        return True

    def object_to_num(self, df=None, col=None):
        """Convert string dtype to float64."""
        if df is not None:
            for column in df:
                if df[column].dtype == object:
                    try:
                        df[column] = df[column].astype('float64')
                    except ValueError:
                        pass
            return df
        elif col is not None:
            try:
                col = col.astype('float64')
            except ValueError:
                pass
            return col

    def optimization_dataframe(
        self, df=pandas.DataFrame(), column_subtypes=dict(), use_object=True
    ):
        """Convert float and int types to minimal byte size dtypes."""
        optimal_types = dict()

        for column in df:

            column_dtype = df[column].dtype

            if column_dtype.name == 'category':
                df[column] = df[column].apply(to_str)
                if 'nan' not in df[column].cat.categories:
                    df[column].cat = df[column].cat.add_categories(['nan'])
                df[column][pandas.isna(df[column])] = 'nan'

            else:
                if column_subtypes:
                    subtype = column_subtypes.get(column, '')
                else:
                    subtype = str()

                check_int = numpy.issubdtype(column_dtype, numpy.integer)
                check_float = numpy.issubdtype(column_dtype, float)
                check_object = numpy.issubdtype(column_dtype, object)

                if check_int or subtype == 'integer' or (
                    check_float and self._is_integer(df[column])
                ):
                    # 'datetime'
                    if subtype == 'datetime':
                        optimal_types.update({column: numpy.datetime64})
                        continue

                    # 'integer'
                    if subtype == 'integer' and check_object:
                        max = np.nanmax(df[column].astype(int))
                        min = np.nanmin(df[column].astype(int))
                    else:
                        max = np.nanmax(df[column])
                        min = np.nanmin(df[column])

                    # 'signed'
                    if max <= self.i8.max and min >= self.i8.min:
                        optimal_types.update({column: 'int8'})
                    elif max <= self.i16.max and min >= self.i16.min:
                        optimal_types.update({column: 'int16'})
                    elif max <= self.i32.max and min >= self.i32.min:
                        optimal_types.update({column: 'int32'})
                    # else:
                        # 'int64'

                elif check_float:
                    # 'float'
                    optimal_types.update({column: 'float32'})

                elif check_object and use_object:
                    # 'object'
                    if subtype == 'datetime':
                        optimal_types.update({column: numpy.datetime64})
                        continue
                    elif subtype in ['binary', 'multi', '']:
                        df[column] = df[column].apply(to_str)
                        df[column][pandas.isna(df[column])] = 'nan'
                        optimal_types.update({column: 'category'})

        df = df.astype(optimal_types, errors='ignore')

        for column in df:
            if df[column].dtype.name == 'category' and 'nan' not in df[column].cat.categories:
                df[column].cat = df[column].cat.add_categories(['nan'])

        return df


def unique_months_(unique_values):
    """Get unique distinct months from Timestamps."""
    unique_values = [
        str(value)
        for value in unique_values
        if str(value) not in ['nan', 'NaN', 'NaT']
    ]
    for datetime_case in ['normal', '%Y%m', '%y%m%d', '%m%Y', '%m%d%y']:
        try:
            if datetime_case == 'normal':
                timestamps = pandas.to_datetime(
                    [value for value in unique_values]
                )
            else:
                timestamps = pandas.to_datetime(
                    [value.zfill(6) for value in unique_values],
                    format=datetime_case
                )
        except Exception:
            continue

    unique_months_numpy = numpy.unique(timestamps.strftime('%Y-%m'))
    unique_months = [month for month in unique_months_numpy if month != 'NaT']

    return unique_months


def get_date_units(unique_values):
    """Return date_unit set that can be unit of unique values."""
    # TODO: Develop
    return ['year', 'year_month', 'year_month_day', 'year_month_day_hour', 'month', 'day', 'hour']


def dataframe_to_csv_gz(
    path, df=pandas.DataFrame(), chunksize=1000000, mode='w',
    index=False, index_label=None, encoding=None
):
    """Save dataframe to csv.gz file by compressing."""
    arange = numpy.arange(0, len(df), chunksize)
    for i, value in enumerate(arange):
        # Write initially.
        if i == 0:
            write_or_append = 'w' if mode == 'w' else 'a'
            header = True if mode == 'w' else False
            left_index = arange[i]
            # If dataset length is not longer than chunk size, then write all.
            if i == len(arange) - 1:
                right_index = None
            # If dataset length is longer than chunk size, then write by chunk size.
            else:
                right_index = arange[i+1]
        # Append after first writing check.
        else:
            write_or_append = 'a'
            header = False
            left_index = arange[i]
            # If dataset length is not longer than chunk size, then append rest of all.
            if i == len(arange) - 1:
                right_index = None
            # If dataset length is same as chunk size, then append by chunk size.
            else:
                right_index = arange[i + 1]

        # Write or append with above settings.
        with gzip.open(path, write_or_append) as f_out:
            f_out.write(
                df.iloc[left_index:right_index, :].to_csv(
                    index=index, index_label=index_label, encoding=encoding, header=header
                ).encode()
            )


def dataframe_concatenation(chunks, column_subtypes=dict()):
    """Concatenate each DataFrame while preserving categorical columns."""
    optimizer = PandasOptimizer()
    try:
        # Iterate on categorical columns common to all chunks
        for column in set.intersection(
                *[
                    set(chunk.select_dtypes(include=['category']).columns)
                    for chunk in chunks
                ]
        ):
            # Generate the union category across chunks for this column
            uc = union_categoricals([chunk[column] for chunk in chunks])
            # Change to union category for all chunks
            for chunk in chunks:
                chunk[column] = pandas.Categorical(
                    chunk[column].values,
                    categories=uc.categories
                )

        data_df = optimizer.optimization_dataframe(
            pandas.concat(chunks, axis=0, ignore_index=True),
            column_subtypes=column_subtypes,
            use_object=True if column_subtypes else False
        )
        del chunks
        gc.collect()
        return data_df
    except ValueError:
        return None


def to_str(x):
    """Return vectorized numpy.array."""
    if isinstance(x, float) and (not numpy.isnan(x)) and x == int(x):
        return str(int(x))
    elif x is None:
        return 'nan'
    else:
        return str(x)

pandas_optimizer = PandasOptimizer()

In [ ]:
pandas_optimizer = PandasOptimizer()

In [ ]:
# accepted 첫번째 10만 건 메모리: 0.267GB
# accepted 두번째 10만 건 메모리: 0.267GB

df_accepted_1st_100000 = pd.read_csv("/content/accepted_2007_to_2018Q4.csv", nrows=100000)
# 메모리 사용량 계산
memory_usage = df_accepted_1st_100000.memory_usage(deep=True).sum() / (1024 ** 3)  # 바이트를 GB로 변환
print(f"DataFrame의 메모리 사용량: {memory_usage:.3f} GB")

df_accepted_2nd_100000 = pd.read_csv("/content/accepted_2007_to_2018Q4.csv", nrows=100000, skiprows=range(1,100001))
# 메모리 사용량 계산
memory_usage = df_accepted_2nd_100000.memory_usage(deep=True).sum() / (1024 ** 3)  # 바이트를 GB로 변환
print(f"DataFrame의 메모리 사용량: {memory_usage:.3f} GB")

<ipython-input-27-1551bc2143ba>:4: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_accepted_1st_100000 = pd.read_csv("/content/accepted_2007_to_2018Q4.csv", nrows=100000)


DataFrame의 메모리 사용량: 0.267 GB


<ipython-input-27-1551bc2143ba>:9: DtypeWarning: Columns (19,59) have mixed types. Specify dtype option on import or set low_memory=False.
  df_accepted_2nd_100000 = pd.read_csv("/content/accepted_2007_to_2018Q4.csv", nrows=100000, skiprows=range(1,100001))


DataFrame의 메모리 사용량: 0.267 GB


In [ ]:
# df_accepted_1st_100000.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# df_accepted_2nd_100000.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,62237680,NaN,6000.0,6000.0,6000.0,36 months,10.99,196.41,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,62257724,NaN,8500.0,8500.0,8500.0,36 months,13.99,290.47,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,62205519,NaN,11525.0,11525.0,11525.0,36 months,8.18,362.11,B,B1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,60772979,NaN,14400.0,14400.0,14400.0,60 months,9.99,305.89,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,62227449,NaN,15000.0,15000.0,15000.0,36 months,12.29,500.30,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# df_accepted_1st_100000.dtypes

id                         int64
member_id                float64
loan_amnt                float64
funded_amnt              float64
funded_amnt_inv          float64
                          ...   
settlement_status         object
settlement_date           object
settlement_amount        float64
settlement_percentage    float64
settlement_term          float64
Length: 151, dtype: object

In [ ]:
# accepted 첫번째 10만 건 최적화 후 메모리: 0.062GB
# accepted 두번째 10만 건 최적화 후 메모리: 0.062GB

df_optimized_1st_100000 = pandas_optimizer.optimization_dataframe(df_accepted_1st_100000)
# 메모리 사용량 계산
memory_usage = df_optimized_1st_100000.memory_usage(deep=True).sum() / (1024 ** 3)  # 바이트를 GB로 변환
print(f"DataFrame의 메모리 사용량: {memory_usage:.3f} GB")

df_optimized_2nd_100000 = pandas_optimizer.optimization_dataframe(df_accepted_2nd_100000)
# 메모리 사용량 계산
memory_usage = df_optimized_2nd_100000.memory_usage(deep=True).sum() / (1024 ** 3)  # 바이트를 GB로 변환
print(f"DataFrame의 메모리 사용량: {memory_usage:.3f} GB")

DataFrame의 메모리 사용량: 0.232 GB
DataFrame의 메모리 사용량: 0.232 GB


In [ ]:
# df_optimized_1st_100000.dtypes

id                         int32
member_id                float64
loan_amnt                  int32
funded_amnt                int32
funded_amnt_inv            int32
                          ...   
settlement_status         object
settlement_date           object
settlement_amount        float32
settlement_percentage    float32
settlement_term          float64
Length: 151, dtype: object

In [ ]:
del df_accepted_1st_100000
del df_accepted_2nd_100000
del df_optimized_1st_100000
del df_optimized_2nd_100000
gc.collect()

53

## 50만개 Chunk씩 Dataframe 읽어와서 각각 concat 진행하기

In [ ]:
# Chunk 단위로 DataFrame을 읽어온 후 optimize_memory 함수를 적용하여 chunks 리스트에 저장
chunk_size = 500000

for index, chunk in enumerate(pd.read_csv("/content/accepted_2007_to_2018Q4.csv", chunksize=chunk_size, low_memory=False)):
    print("===", index, "===")
    if index==0:
        df_accepted = pandas_optimizer.optimization_dataframe(chunk)
        print(df_accepted.shape, df_accepted.memory_usage(deep=True).sum() / (1024 ** 3))
    else:
        optimized_chunk = pandas_optimizer.optimization_dataframe(chunk)
        dfs = [df_accepted, optimized_chunk]
        for column in set.intersection(
            *[
                set(df.select_dtypes(include=['category']).columns)
                for df in dfs
            ]
        ):
            uc = union_categoricals([df[column] for df in dfs])
            for df in dfs:
                df[column] = pandas.Categorical(
                    df[column].values,
                    categories=uc.categories
                )
        df_accepted = pandas_optimizer.optimization_dataframe(
            pd.concat(
                dfs,
                ignore_index=True
            )
        )
        print(df_accepted.shape, df_accepted.memory_usage(deep=True).sum() / (1024 ** 3))
        del optimized_chunk
        del chunk
        del dfs

gc.collect()

=== 0 ===


<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:191: RuntimeWarning: All-NaN axis encountered
  max = np.nanmax(df[column])
<ipython-input-2-8535f5d4e586>:192: RuntimeWarning: All-NaN axis encountered
  min = np.nanmin(df[column])
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documenta

(500000, 151) 0.5248473398387432
=== 1 ===


<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:191: RuntimeWarning: All-NaN axis encountered
  max = np.nanmax(df[column])
<ipython-input-2-8535f5d4e586>:192: RuntimeWarning: All-NaN axis encountered
  min = np.nanmin(df[column])
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documenta

(1000000, 151) 1.046936976723373
=== 2 ===


...

<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:191: RuntimeWarning: All-NaN axis encountered
  max = np.nanmax(df[column])
<ipython-input-2-8535f5d4e586>:192: RuntimeWarning: All-NaN axis encountered
  min = np.nanmin(df[column])
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documenta

(2000000, 151) 2.1565730161964893
=== 4 ===


<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:191: RuntimeWarning: All-NaN axis encountered
  max = np.nanmax(df[column])
<ipython-input-2-8535f5d4e586>:192: RuntimeWarning: All-NaN axis encountered
  min = np.nanmin(df[column])
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'
<ipython-input-2-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documenta

(2260701, 151) 2.411354393698275


0

## 위 Concatenation이 느린 이유?
## memory 최적화를 진행하지 않아도 되는 데이터들과 변수들이 있음에도 모든 경우를 계속해서 최적화하고 있기 때문
## 그리고, 변수들끼리 메모리 최적화 과정을 진행할 때 Multi Threading을 활용하면 훨씬 더 최적화가 잘 될 예정.

In [ ]:
# 결과 확인
print(df_accepted.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: category(38), float32(28), float64(85)
memory usage: 2.0 GB
None


In [ ]:
# df_accepted.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.990000,123.029999,C,C4,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.990000,820.280029,C,C1,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.780000,432.660004,B,B4,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.850000,829.900024,C,C5,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.450001,289.910004,F,F1,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN


In [ ]:
# df_accepted.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.990000,123.029999,C,C4,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.990000,820.280029,C,C1,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.780000,432.660004,B,B4,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.850000,829.900024,C,C5,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.450001,289.910004,F,F1,...,NaN,NaN,Cash,N,nan,nan,nan,NaN,NaN,NaN


In [ ]:
# df_accepted.shape

(2260701, 151)

In [ ]:
for index, column in enumerate(df_accepted):
    print(index, column)
    try:
        df_accepted[[column]].to_parquet('abc.parquet', index=False)
    except Exception:
        print(column, df_accepted[[column]].dtypes[0])

0 id
1 member_id
2 loan_amnt
3 funded_amnt
4 funded_amnt_inv
5 term
6 int_rate
7 installment
8 grade
9 sub_grade
10 emp_title
11 emp_length
12 home_ownership
13 annual_inc
14 verification_status
15 issue_d
16 loan_status
17 pymnt_plan
18 url
19 desc
20 purpose
21 title
22 zip_code
23 addr_state
24 dti
25 delinq_2yrs
26 earliest_cr_line
27 fico_range_low
28 fico_range_high
29 inq_last_6mths
30 mths_since_last_delinq
31 mths_since_last_record
32 open_acc
33 pub_rec
34 revol_bal
35 revol_util
36 total_acc
37 initial_list_status
38 out_prncp
39 out_prncp_inv
40 total_pymnt
41 total_pymnt_inv
42 total_rec_prncp
43 total_rec_int
44 total_rec_late_fee
45 recoveries
46 collection_recovery_fee
47 last_pymnt_d
48 last_pymnt_amnt
49 next_pymnt_d
50 last_credit_pull_d
51 last_fico_range_high
52 last_fico_range_low
53 collections_12_mths_ex_med
54 mths_since_last_major_derog
55 policy_code
56 application_type
57 annual_inc_joint
58 dti_joint
59 verification_status_joint
60 acc_now_delinq
61 tot_col

In [ ]:
# 1.6GB -> 319MB Parquet로 저장완료
# 319MB로 저장 완료
df_accepted.iloc[:,0:].to_parquet('accepted.parquet', index=False)

In [ ]:
# !ls -lah

total 5.2G
drwxr-xr-x 1 root root 4.0K Aug  6 08:02 .
drwxr-xr-x 1 root root 4.0K Aug  6 01:13 ..
-rw-r--r-- 1 root root 319M Aug  6 08:01 abcd.parquet
-rw-r--r-- 1 root root  96K Aug  6 08:00 abc.parquet
drwxr-xr-x 2 root root 4.0K Aug  6 01:18 accepted_2007_to_2018q4.csv
-rw-r--r-- 1 root root 1.6G Dec 17  2019 accepted_2007_to_2018Q4.csv
-rw-r--r-- 1 root root 319M Aug  6 08:02 accepted.parquet
drwxr-xr-x 4 root root 4.0K Aug  3 13:44 .config
-rw-r--r-- 1 root root 3.0K Aug  6 07:43 haha.parquet
-rw-r--r-- 1 root root 1.3G Dec 17  2019 lending-club.zip
drwxr-xr-x 2 root root 4.0K Aug  6 01:18 rejected_2007_to_2018q4.csv
-rw-r--r-- 1 root root 1.7G Dec 17  2019 rejected_2007_to_2018Q4.csv
drwxr-xr-x 1 root root 4.0K Aug  3 13:45 sample_data


In [ ]:
# pd.read_parquet는 터지지도 않고,
# Reading하는데 걸리는 시간은 14.3초
%%time
df_accepted = pd.read_parquet('accepted.parquet')

CPU times: user 12.3 s, sys: 4.63 s, total: 16.9 s
Wall time: 14.3 s


In [ ]:
# 메모리는 그대로 유지되어서 2.0GB로 최적화 완료.
df_accepted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: category(38), float32(28), float64(85)
memory usage: 2.0 GB


In [ ]:
# Chunk 단위로 DataFrame을 읽어온 후 optimize_memory 함수를 적용하여 chunks 리스트에 저장
chunk_size = 1000000

for index, chunk in enumerate(pd.read_csv("/content/rejected_2007_to_2018Q4.csv", chunksize=chunk_size, low_memory=False)):
    print("===", index, "===")
    if index==0:
        df_rejected = pandas_optimizer.optimization_dataframe(chunk)
        print(df_rejected.shape, df_rejected.memory_usage(deep=True).sum() / (1024 ** 3))
    else:
        optimized_chunk = pandas_optimizer.optimization_dataframe(chunk)
        dfs = [df_rejected, optimized_chunk]
        for column in set.intersection(
            *[
                set(df.select_dtypes(include=['category']).columns)
                for df in dfs
            ]
        ):
            uc = union_categoricals([df[column] for df in dfs])
            for df in dfs:
                df[column] = pandas.Categorical(
                    df[column].values,
                    categories=uc.categories
                )
        df_rejected = pandas_optimizer.optimization_dataframe(
            pd.concat(
                dfs,
                ignore_index=True
            )
        )
        print(df_rejected.shape, df_rejected.memory_usage(deep=True).sum() / (1024 ** 3))
        del optimized_chunk
        del chunk
        del dfs

gc.collect()

=== 0 ===


<ipython-input-15-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'


(1000000, 9) 0.0325021268799901
=== 1 ===


<ipython-input-15-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'


(2000000, 9) 0.06021258421242237
=== 2 ===


...

<ipython-input-15-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'


(27000000, 9) 0.8741419464349747
=== 27 ===


<ipython-input-15-8535f5d4e586>:215: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column][pandas.isna(df[column])] = 'nan'


(27648741, 9) 0.8947467971593142


0

In [ ]:
# 예상되었던 메모리 10.5 GB => 0.904 GB로 최적화 완료!
df_rejected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27648741 entries, 0 to 27648740
Data columns (total 9 columns):
 #   Column                Dtype   
---  ------                -----   
 0   Amount Requested      float32 
 1   Application Date      category
 2   Loan Title            category
 3   Risk_Score            float64 
 4   Debt-To-Income Ratio  category
 5   Zip Code              category
 6   State                 category
 7   Employment Length     category
 8   Policy Code           float64 
dtypes: category(6), float32(1), float64(2)
memory usage: 904.3 MB


In [ ]:
df_rejected.shape

(27648741, 9)

In [ ]:
# 1.7GB CSV -> 198MB Parquet로 저장완료
# 0.904 GB memory -> 198MB Parquet로 저장완료
%%time
df_rejected.iloc[:,0:].to_parquet('rejected.parquet', index=False)

CPU times: user 7.77 s, sys: 1.32 s, total: 9.08 s
Wall time: 8.86 s


In [ ]:
!ls -lah

total 4.7G
drwxr-xr-x 1 root root 4.0K Aug  7 11:03 .
drwxr-xr-x 1 root root 4.0K Aug  7 10:05 ..
drwxr-xr-x 2 root root 4.0K Aug  7 10:09 accepted_2007_to_2018q4.csv
-rw-r--r-- 1 root root 1.6G Dec 17  2019 accepted_2007_to_2018Q4.csv
drwxr-xr-x 4 root root 4.0K Aug  3 13:44 .config
-rw-r--r-- 1 root root 1.3G Dec 17  2019 lending-club.zip
drwxr-xr-x 2 root root 4.0K Aug  7 10:09 rejected_2007_to_2018q4.csv
-rw-r--r-- 1 root root 1.7G Dec 17  2019 rejected_2007_to_2018Q4.csv
-rw-r--r-- 1 root root 198M Aug  7 11:03 rejected.parquet
drwxr-xr-x 1 root root 4.0K Aug  3 13:45 sample_data


In [ ]:
# pd.read_parquet는 터지지도 않고,
# Reading하는데 걸리는 시간은 5초
%%time
df_rejected = pd.read_parquet('rejected.parquet')

CPU times: user 4.68 s, sys: 4.74 s, total: 9.43 s
Wall time: 6.22 s


In [ ]:
# 메모리는 그대로 유지되어서 0.904 MB로 최적화 완료.
df_rejected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27648741 entries, 0 to 27648740
Data columns (total 9 columns):
 #   Column                Dtype   
---  ------                -----   
 0   Amount Requested      float32 
 1   Application Date      category
 2   Loan Title            category
 3   Risk_Score            float64 
 4   Debt-To-Income Ratio  category
 5   Zip Code              category
 6   State                 category
 7   Employment Length     category
 8   Policy Code           float64 
dtypes: category(6), float32(1), float64(2)
memory usage: 904.3 MB
